In [1]:
# 05_sensitivity.ipynb
# Regulatory-Constrained Fairness Score for Credit Scoring
# Notebook 05 - Sensitivity and statistical-significance analysis.
#
# Hypothesis under test:
#   H1 : on a given dataset, at least one fairness-mitigation technique attains a
#        strictly higher RCFS than the unmitigated baseline.
# H1 is reported as a point outcome AND as a statistical claim: a technique is only
# credited with beating the baseline when the paired RCFS difference (technique minus
# baseline) is positive and its 95% bootstrap interval excludes zero.
#
# Studies:
#   A. k_star sweep                  - does the best technique depend on k*? Also
#                                       reports the compliance-dominant boundary c_expl*
#                                       (where Compliance = c_perf) and how it moves with k*.
#   B. Harmonic-weight sweep         - reweight c_fair vs c_expl.
#   C. Gamma sweep and rank stability- are gamma-induced reorderings real or ties?
#   D. Paired bootstrap (fixed split)- 95% CI for RCFS(tech) - RCFS(baseline), all
#                                       techniques, on the frozen test split.
#   E. Paired difference over K random splits - mean technique-minus-baseline RCFS and
#                                       its spread across 10 independent 60/20/20 splits,
#                                       with the fraction of splits the technique wins.
# Fairness uses the favourable (non-default) outcome, worst-case across sensitive
# attributes; thresholds are validation-tuned. No paths are printed.

import os
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(TAB_DIR, exist_ok=True)

RANDOM_STATE = 42
SHAP_SAMPLE = 1000
BASE_KSTAR = 10
GAMMA_DEFAULT = 0.5
W_FAIR, W_EXPL = 1.0, 1.0
N_BOOT = 1000
N_SPLITS = 10
XGB_PARAMS = dict(n_estimators=400, max_depth=5, learning_rate=0.05,
                  subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
                  random_state=RANDOM_STATE, n_jobs=-1)
MULTI_SPLIT_TECHS = ["reweighing", "reject_option"]  # vs baseline, fast XGBoost family
ALL_TECHS = ["baseline", "reweighing", "reject_option"]

with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)
DATASETS = [("German Credit", "german"), ("Taiwan CC Default", "taiwan"),
            ("Credit Card Approval", "credit_card_approval")]
PRIVILEGED = {"sex": "male", "age_group": "old", "education": ["graduate", "university"]}

# --- Fairness / explainability / RCFS helpers (favourable outcome, worst-case) ---
def priv_mask_for(series, privileged):
    if isinstance(privileged, (list, tuple)):
        return series.isin(list(privileged)).values
    return (series == privileged).values


def c_fair_worstcase(df_sens, sens_cols, privileged, fav, y_pred):
    best = None
    y_pred = np.asarray(y_pred)
    for a in sens_cols:
        s = df_sens[a]
        pm = priv_mask_for(s, privileged[a]); um = (~pm) & s.notna().values
        fav_mask = (y_pred == fav)
        r_p = fav_mask[pm].mean() if pm.sum() else np.nan
        r_u = fav_mask[um].mean() if um.sum() else np.nan
        if not (np.isfinite(r_p) and np.isfinite(r_u)) or r_p == 0:
            continue
        di, spd = r_u / r_p, r_u - r_p
        score = 0.5 * min(1.0, di) + 0.5 * (1.0 - abs(spd))
        best = score if best is None else min(best, score)
    return best


def shap_neff(model, X, sample=SHAP_SAMPLE, seed=RANDOM_STATE):
    if len(X) > sample:
        X = X.sample(sample, random_state=seed)
    try:
        sv = shap.TreeExplainer(model).shap_values(X)
    except Exception:
        sv = shap.Explainer(model, X)(X).values
    if isinstance(sv, list):
        sv = sv[-1]
    mean_abs = np.abs(sv).mean(axis=0); total = mean_abs.sum()
    if total == 0:
        return np.nan
    p = mean_abs / total; p = p[p > 0]
    if len(p) <= 1:
        return 1.0
    return float(np.exp(-np.sum(p * np.log(p))))


def c_expl_from_neff(n_eff, k_star):
    return np.nan if not np.isfinite(n_eff) else float(min(1.0, k_star / n_eff))


def harmonic(cs, ws):
    pairs = [(c, w) for c, w in zip(cs, ws) if np.isfinite(c) and c > 0]
    if not pairs:
        return np.nan
    return sum(w for _, w in pairs) / sum(w / c for c, w in pairs)


def rcfs(c_perf, comp, gamma):
    return np.nan if not np.isfinite(comp) else (1 - gamma) * c_perf + gamma * comp


def spw_of(y):
    n_pos = int(y.sum()); n_neg = int((y == 0).sum())
    return (n_neg / n_pos) if n_pos > 0 else 1.0


def reweighing_weights(y, s):
    w = np.ones(len(y), float); n = len(y)
    for yy in np.unique(y):
        for ss in pd.unique(s):
            m = (y == yy) & (s.values == ss)
            if m.sum():
                w[m] = ((y == yy).mean() * (s.values == ss).mean()) / (m.sum() / n)
    return w


def reject_thresholds(proba_val, s_val, rate):
    return {g: (np.quantile(proba_val[s_val.values == g], 1 - rate)
                if (s_val.values == g).any() else 0.5) for g in pd.unique(s_val)}


def get_Xy(df, key):
    meta = META[key]
    drop = [meta["target_col"]] + meta["sensitive_cols"]
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    return X, df[meta["target_col"]].astype(int)


def load_split(key, part):
    return pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_{part}.parquet"))

# Cache base components from the FROZEN split (studies A-D).
cache = {}
for display_name, key in DATASETS:
    meta = META[key]; prim = meta["sensitive_cols"][0]; fav = meta["favourable_label"]
    tr, val, te = load_split(key, "train"), load_split(key, "val"), load_split(key, "test")
    X_tr, y_tr = get_Xy(tr, key); X_val, _ = get_Xy(val, key); X_te, y_te = get_Xy(te, key)
    X_val = X_val[X_tr.columns]; X_te = X_te[X_tr.columns]
    s_tr, s_val, s_te = tr[prim], val[prim], te[prim]
    rate = float(y_tr.mean())
    for tech in ALL_TECHS:
        model = XGBClassifier(scale_pos_weight=spw_of(y_tr), **XGB_PARAMS)
        if tech == "reweighing":
            model.fit(X_tr, y_tr, sample_weight=reweighing_weights(y_tr.values, s_tr))
        else:
            model.fit(X_tr, y_tr)
        proba_val = model.predict_proba(X_val)[:, 1]
        proba_te = model.predict_proba(X_te)[:, 1]
        if tech == "reject_option":
            tmap = reject_thresholds(proba_val, s_val, rate)
            y_pred = np.array([int(p >= tmap.get(g, 0.5)) for p, g in zip(proba_te, s_te.values)])
        else:
            thr = np.quantile(proba_val, 1 - rate)
            y_pred = (proba_te >= thr).astype(int)
            tmap = None
        auc = roc_auc_score(y_te.values, proba_te)
        cache[(display_name, tech)] = dict(
            c_perf=max(0.0, (auc - 0.5) / 0.5), n_eff=shap_neff(model, X_te),
            proba=proba_te, y_true=y_te.values, s_te=s_te.reset_index(drop=True),
            te_sens=te[meta["sensitive_cols"]].reset_index(drop=True),
            sens_cols=meta["sensitive_cols"], fav=fav, rate=rate, tmap=tmap)
    print(f"[{display_name}] frozen-split components cached")


def score_cached(display_name, tech, k_star, w_fair, w_expl, gamma):
    c = cache[(display_name, tech)]
    y_pred = (c["proba"] >= np.quantile(c["proba"], 1 - c["rate"])).astype(int) \
        if c["tmap"] is None else \
        np.array([int(p >= c["tmap"].get(g, 0.5)) for p, g in zip(c["proba"], c["s_te"].values)])
    c_fair = c_fair_worstcase(c["te_sens"], c["sens_cols"], PRIVILEGED, c["fav"], y_pred)
    c_expl = c_expl_from_neff(c["n_eff"], k_star)
    return rcfs(c["c_perf"], harmonic([c_fair, c_expl], [w_fair, w_expl]), gamma)

# --- Study A: k_star sweep + compliance-dominant boundary ---
kstars = [4, 6, 8, 10, 12, 15, 20]
rowsA, rowsA2 = [], []
for ks in kstars:
    for display_name, key in DATASETS:
        sc = {t: score_cached(display_name, t, ks, W_FAIR, W_EXPL, GAMMA_DEFAULT) for t in ALL_TECHS}
        best = max(sc, key=sc.get)
        rowsA.append({"k_star": ks, "dataset": display_name, "best": best,
                      "beats_baseline": sc[best] > sc["baseline"] + 1e-9,
                      **{f"RCFS_{t}": round(sc[t], 4) for t in ALL_TECHS}})
studyA = pd.DataFrame(rowsA)
studyA.to_csv(os.path.join(TAB_DIR, "05_sensitivity_kstar.csv"), index=False)
# Compliance-dominant boundary c_expl* (Compliance == c_perf) at baseline per dataset.
for display_name, key in DATASETS:
    c = cache[(display_name, "baseline")]
    y_pred = (c["proba"] >= np.quantile(c["proba"], 1 - c["rate"])).astype(int)
    c_fair = c_fair_worstcase(c["te_sens"], c["sens_cols"], PRIVILEGED, c["fav"], y_pred)
    c_perf = c["c_perf"]
    denom = (2.0 / c_perf) - (1.0 / c_fair) if c_fair and c_perf else np.nan
    c_expl_star = (1.0 / denom) if (np.isfinite(denom) and denom > 0) else np.nan
    c_expl_obs = c_expl_from_neff(c["n_eff"], BASE_KSTAR)
    rowsA2.append({"dataset": display_name, "c_perf": round(c_perf, 4),
                   "c_fair": round(c_fair, 4), "n_eff": round(c["n_eff"], 3),
                   "c_expl_obs(k*=10)": round(c_expl_obs, 4),
                   "c_expl_boundary": round(c_expl_star, 4) if np.isfinite(c_expl_star) else np.nan,
                   "regime": ("compliance-dominant" if c_expl_obs > c_expl_star
                              else "performance-dominant") if np.isfinite(c_expl_star) else "n/a"})
studyA2 = pd.DataFrame(rowsA2)
studyA2.to_csv(os.path.join(TAB_DIR, "05_compliance_boundary.csv"), index=False)
print("Study A - k* sweep (best technique):")
print(studyA.to_string(index=False))
print("Study A - compliance-dominant boundary:")
print(studyA2.to_string(index=False))

# --- Study B: harmonic-weight sweep ---
rowsB = []
for wf, we in [(2.0, 1.0), (1.5, 1.0), (1.0, 1.0), (1.0, 1.5), (1.0, 2.0)]:
    for display_name, key in DATASETS:
        sc = {t: score_cached(display_name, t, BASE_KSTAR, wf, we, GAMMA_DEFAULT) for t in ALL_TECHS}
        best = max(sc, key=sc.get)
        rowsB.append({"w_fair": wf, "w_expl": we, "dataset": display_name,
                      "best": best, "beats_baseline": sc[best] > sc["baseline"] + 1e-9})
studyB = pd.DataFrame(rowsB)
studyB.to_csv(os.path.join(TAB_DIR, "05_sensitivity_weights.csv"), index=False)
print("Study B - harmonic weights:")
print(studyB.to_string(index=False))

# --- Study C: gamma sweep rank stability ---
gammas = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
rowsC = []
for g in gammas:
    for display_name, key in DATASETS:
        sc = {t: score_cached(display_name, t, BASE_KSTAR, W_FAIR, W_EXPL, g) for t in ALL_TECHS}
        order = sorted(sc, key=sc.get, reverse=True)
        gap = sc[order[0]] - sc[order[1]]
        rowsC.append({"gamma": g, "dataset": display_name, "best": order[0],
                      "runner_up": order[1], "top2_gap": round(gap, 4)})
studyC = pd.DataFrame(rowsC)
studyC.to_csv(os.path.join(TAB_DIR, "05_sensitivity_gamma.csv"), index=False)
print("Study C - gamma sweep (best, runner-up, gap):")
print(studyC.to_string(index=False))

# --- Study D: paired bootstrap of RCFS(tech) - RCFS(baseline) on the frozen split ---
def bootstrap_paired(display_name, techs, n_boot=N_BOOT, seed=RANDOM_STATE):
    base = cache[(display_name, "baseline")]
    rng = np.random.default_rng(seed)
    n = len(base["y_true"])
    out = {t: [] for t in techs}
    c_expl = {t: c_expl_from_neff(cache[(display_name, t)]["n_eff"], BASE_KSTAR)
              for t in (["baseline"] + techs)}
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(base["y_true"][idx])) < 2:
            continue

        def rcfs_of(t):
            c = cache[(display_name, t)]
            yb, pb = c["y_true"][idx], c["proba"][idx]
            sens_b = c["te_sens"].iloc[idx]; s_b = c["s_te"].iloc[idx]
            cp = max(0.0, (roc_auc_score(yb, pb) - 0.5) / 0.5)
            if c["tmap"] is None:
                yhat = (pb >= np.quantile(pb, 1 - c["rate"])).astype(int)
            else:
                tm = reject_thresholds(pb, s_b, c["rate"])
                yhat = np.array([int(p >= tm.get(g, 0.5)) for p, g in zip(pb, s_b.values)])
            cf = c_fair_worstcase(sens_b, c["sens_cols"], PRIVILEGED, c["fav"], yhat)
            return rcfs(cp, harmonic([cf, c_expl[t]], [W_FAIR, W_EXPL]), GAMMA_DEFAULT)

        rb = rcfs_of("baseline")
        for t in techs:
            out[t].append(rcfs_of(t) - rb)
    return out

rowsD = []
for display_name, key in DATASETS:
    diffs = bootstrap_paired(display_name, [t for t in ALL_TECHS if t != "baseline"])
    for t, arr in diffs.items():
        a = np.array([v for v in arr if np.isfinite(v)])
        lo, hi = np.percentile(a, 2.5), np.percentile(a, 97.5)
        rowsD.append({"dataset": display_name, "technique": t,
                      "mean_diff": round(a.mean(), 4), "CI_low": round(lo, 4),
                      "CI_high": round(hi, 4), "significant": bool(lo > 0)})
studyD = pd.DataFrame(rowsD)
studyD.to_csv(os.path.join(TAB_DIR, "05_paired_bootstrap_diff.csv"), index=False)
print("Study D - paired bootstrap, RCFS(tech) - RCFS(baseline), frozen split:")
print(studyD.to_string(index=False))

# --- Study E: paired difference across K independent random splits ---
RAW = {k: pd.read_parquet(os.path.join(PROCESSED_DIR, f"{k}.parquet")) for _, k in DATASETS}
SPECS = {"german": (["sex", "age_group"], ["personal_status_sex"]),
         "taiwan": (["sex", "education"], ["SEX", "EDUCATION"]),
         "credit_card_approval": (["sex"], ["ID", "CODE_GENDER", "target_30dpd", "target_60dpd"])}


def prep_split(df, sens_cols, drop_cols, seed):
    df = df.copy(); y = df["target"].astype(int); sens = df[sens_cols].copy()
    feat = df.drop(columns=["target"] + sens_cols + drop_cols, errors="ignore")
    num = feat.select_dtypes(include=[np.number]).columns.tolist()
    cat = feat.select_dtypes(exclude=[np.number]).columns.tolist()
    i_trv, i_te = train_test_split(feat.index, test_size=0.20, random_state=seed, stratify=y)
    i_tr, i_val = train_test_split(i_trv, test_size=0.25, random_state=seed, stratify=y.loc[i_trv])
    def build(idx, fit=None):
        raw = feat.loc[idx].copy()
        if num:
            raw[num] = raw[num].fillna(fit["med"] if fit else raw[num].median())
            scl = fit["scl"] if fit else StandardScaler().fit(raw[num])
            np_ = pd.DataFrame(scl.transform(raw[num]), columns=num, index=raw.index)
        else:
            scl = None; np_ = pd.DataFrame(index=raw.index)
        if cat:
            d = pd.get_dummies(raw[cat].astype("object").fillna("missing"))
        else:
            d = pd.DataFrame(index=raw.index)
        return np_, d, scl, (fit["med"] if fit else (raw[num].median() if num else None))
    tr_n, tr_c, scl, med = build(i_tr)
    fit = {"scl": scl, "med": med}
    parts = {}
    train_cols = tr_c.columns
    for name, idx, nc_cc in [("tr", i_tr, (tr_n, tr_c)), ("val", i_val, None), ("te", i_te, None)]:
        if nc_cc is None:
            nc, cc, _, _ = build(idx, fit)
        else:
            nc, cc = nc_cc
        cc = cc.reindex(columns=train_cols, fill_value=0)
        X = pd.concat([nc, cc], axis=1)
        parts[name] = (X, y.loc[idx], sens.loc[idx])
    return parts

rowsE = []
for display_name, key in DATASETS:
    sens_cols, drop_cols = SPECS[key]; prim = sens_cols[0]; fav = 0
    per_tech = {t: [] for t in MULTI_SPLIT_TECHS}
    for seed in range(N_SPLITS):
        P = prep_split(RAW[key], sens_cols, drop_cols, 100 + seed)
        (X_tr, y_tr, sg_tr) = P["tr"]; (X_val, y_val, _) = P["val"]; (X_te, y_te, sg_te) = P["te"]
        X_val = X_val[X_tr.columns]; X_te = X_te[X_tr.columns]
        s_tr, s_val, s_te = sg_tr[prim], P["val"][2][prim], sg_te[prim]
        rate = float(y_tr.mean())
        comp = {}
        for tech in ["baseline"] + MULTI_SPLIT_TECHS:
            m = XGBClassifier(scale_pos_weight=spw_of(y_tr), **XGB_PARAMS)
            if tech == "reweighing":
                m.fit(X_tr, y_tr, sample_weight=reweighing_weights(y_tr.values, s_tr))
            else:
                m.fit(X_tr, y_tr)
            pv = m.predict_proba(X_val)[:, 1]; pt = m.predict_proba(X_te)[:, 1]
            if tech == "reject_option":
                tm = reject_thresholds(pv, s_val, rate)
                yh = np.array([int(p >= tm.get(g, 0.5)) for p, g in zip(pt, s_te.values)])
            else:
                yh = (pt >= np.quantile(pv, 1 - rate)).astype(int)
            cp = max(0.0, (roc_auc_score(y_te.values, pt) - 0.5) / 0.5)
            cf = c_fair_worstcase(sg_te, sens_cols, PRIVILEGED, fav, yh)
            ce = c_expl_from_neff(shap_neff(m, X_te), BASE_KSTAR)
            comp[tech] = rcfs(cp, harmonic([cf, ce], [W_FAIR, W_EXPL]), GAMMA_DEFAULT)
        for tech in MULTI_SPLIT_TECHS:
            per_tech[tech].append(comp[tech] - comp["baseline"])
    for tech in MULTI_SPLIT_TECHS:
        a = np.array(per_tech[tech])
        rowsE.append({"dataset": display_name, "technique": tech,
                      "mean_diff": round(a.mean(), 4), "sd_diff": round(a.std(ddof=1), 4),
                      "win_rate": round((a > 0).mean(), 3), "n_splits": N_SPLITS})
    print(f"[{display_name}] across-split paired differences computed")
studyE = pd.DataFrame(rowsE)
studyE.to_csv(os.path.join(TAB_DIR, "05_multisplit_diff.csv"), index=False)
print("Study E - RCFS(tech) - RCFS(baseline) across", N_SPLITS, "random splits:")
print(studyE.to_string(index=False))
print("H1 is credited only where Study D CI excludes 0 and Study E win_rate is high.")
print("Notebook 05 complete.")


/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[German Credit] frozen-split components cached


[Taiwan CC Default] frozen-split components cached


[Credit Card Approval] frozen-split components cached
Study A - k* sweep (best technique):
 k_star              dataset          best  beats_baseline  RCFS_baseline  RCFS_reweighing  RCFS_reject_option
      4        German Credit    reweighing            True         0.4015           0.4056              0.4027
      4    Taiwan CC Default    reweighing            True         0.4701           0.4711              0.4702
      4 Credit Card Approval    reweighing            True         0.3455           0.3493              0.3454
      6        German Credit    reweighing            True         0.4500           0.4543              0.4525
      6    Taiwan CC Default    reweighing            True         0.5417           0.5428              0.5419
      6 Credit Card Approval    reweighing            True         0.4059           0.4104              0.4056
      8        German Credit    reweighing            True         0.4926           0.4971              0.4965
      8    Taiwan CC 

Study B - harmonic weights:
 w_fair  w_expl              dataset          best  beats_baseline
    2.0     1.0        German Credit reject_option            True
    2.0     1.0    Taiwan CC Default    reweighing            True
    2.0     1.0 Credit Card Approval    reweighing            True
    1.5     1.0        German Credit reject_option            True
    1.5     1.0    Taiwan CC Default    reweighing            True
    1.5     1.0 Credit Card Approval    reweighing            True
    1.0     1.0        German Credit reject_option            True
    1.0     1.0    Taiwan CC Default    reweighing            True
    1.0     1.0 Credit Card Approval    reweighing            True
    1.0     1.5        German Credit    reweighing            True
    1.0     1.5    Taiwan CC Default    reweighing            True
    1.0     1.5 Credit Card Approval    reweighing            True
    1.0     2.0        German Credit    reweighing            True
    1.0     2.0    Taiwan CC Defau

Study D - paired bootstrap, RCFS(tech) - RCFS(baseline), frozen split:
             dataset     technique  mean_diff  CI_low  CI_high  significant
       German Credit    reweighing     0.0031 -0.0110   0.0199        False
       German Credit reject_option     0.0016 -0.0104   0.0135        False
   Taiwan CC Default    reweighing     0.0011 -0.0019   0.0042        False
   Taiwan CC Default reject_option     0.0003 -0.0013   0.0022        False
Credit Card Approval    reweighing     0.0054 -0.0002   0.0115        False
Credit Card Approval reject_option     0.0008  0.0001   0.0017         True


[German Credit] across-split paired differences computed


[Taiwan CC Default] across-split paired differences computed


[Credit Card Approval] across-split paired differences computed
Study E - RCFS(tech) - RCFS(baseline) across 10 random splits:
             dataset     technique  mean_diff  sd_diff  win_rate  n_splits
       German Credit    reweighing     0.0044   0.0087       0.7        10
       German Credit reject_option     0.0010   0.0042       0.6        10
   Taiwan CC Default    reweighing    -0.0000   0.0030       0.7        10
   Taiwan CC Default reject_option    -0.0001   0.0006       0.6        10
Credit Card Approval    reweighing    -0.0014   0.0024       0.4        10
Credit Card Approval reject_option     0.0006   0.0006       0.8        10
H1 is credited only where Study D CI excludes 0 and Study E win_rate is high.
Notebook 05 complete.
